In [ ]:
import sys
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
OUT_DIR = Path('/kaggle/working')

print('Contents of /kaggle/input:', sorted(p.name for p in INPUT_ROOT.iterdir()))


_SKIP_DIR_NAMES = {'train_series', 'test_series'}


def find_dir_containing(marker_relpath, root=INPUT_ROOT, max_depth=8):
    """Breadth-first search under /kaggle/input for a directory containing
    `marker_relpath`, pruning train_series/test_series before descending into them
    (they hold hundreds of thousands of per-study/per-series directories and none of
    our markers live inside them -- a naive search enumerates the whole tree instead,
    confirmed costing ~8 minutes per run without this)."""
    frontier = [root]
    for _ in range(max_depth + 1):
        next_frontier = []
        for candidate in frontier:
            if (candidate / marker_relpath).exists():
                return candidate
            try:
                children = [c for c in candidate.iterdir() if c.is_dir() and c.name not in _SKIP_DIR_NAMES]
            except OSError:
                continue
            next_frontier.extend(children)
        frontier = next_frontier
    raise FileNotFoundError(
        f'Could not find a directory containing {marker_relpath!r} under {root} '
        f'(searched up to depth {max_depth}). Top-level contents: '
        f'{sorted(p.name for p in root.iterdir())}'
    )


SRC_DATASET = find_dir_containing('rsna_knee/__init__.py')
print('Resolved SRC_DATASET:', SRC_DATASET)
sys.path.insert(0, str(SRC_DATASET))

COMPETITION_ROOT = find_dir_containing('sample_submission.csv')
print('Resolved COMPETITION_ROOT:', COMPETITION_ROOT)

In [ ]:
import pandas as pd

from rsna_knee.data.series import assign_slots
from rsna_knee.data.dicom_io import SERIES_SLOTS

train_series_csv = pd.read_csv(COMPETITION_ROOT / 'train_series.csv')
train_series_dir = COMPETITION_ROOT / 'train_series'

print(f'Scanning {len(train_series_csv)} series headers (stop_before_pixels, no pixel decode)...')
series_with_slots = assign_slots(train_series_csv, train_series_dir)
series_with_slots.to_csv(OUT_DIR / 'train_series_with_slots.csv', index=False)
print('Done. Weighting value counts:')
print(series_with_slots['weighting'].value_counts())
print()
print('Slot value counts (None = did not classify into any of the 6 canonical slots):')
print(series_with_slots['slot'].value_counts(dropna=False))

In [ ]:
from rsna_knee.data.series import slot_presence_mask

coverage = {slot: 0 for slot in SERIES_SLOTS}
n_studies = 0
for study_id, group in series_with_slots.groupby('StudyInstanceUID'):
    mask = slot_presence_mask(group['slot'].tolist())
    for slot, present in mask.items():
        if present:
            coverage[slot] += 1
    n_studies += 1

print(f'Per-slot presence coverage across {n_studies} training studies:')
for slot in SERIES_SLOTS:
    pct = 100 * coverage[slot] / n_studies
    print(f'  {slot:16s} {coverage[slot]:5d} / {n_studies}  ({pct:.1f}%)')

In [ ]:
train_labels = pd.read_csv(COMPETITION_ROOT / 'train.csv')
label_cols = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
              'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
gold_mask = train_labels[label_cols].notna().any(axis=1)
gold_ids = set(train_labels.loc[gold_mask, 'StudyInstanceUID'])
print(f'{len(gold_ids)} gold studies')

gold_slots = series_with_slots[series_with_slots['StudyInstanceUID'].isin(gold_ids)]
gold_coverage = {slot: 0 for slot in SERIES_SLOTS}
for study_id, group in gold_slots.groupby('StudyInstanceUID'):
    mask = slot_presence_mask(group['slot'].tolist())
    for slot, present in mask.items():
        if present:
            gold_coverage[slot] += 1

print(f'Per-slot presence coverage across the {len(gold_ids)} gold studies specifically:')
for slot in SERIES_SLOTS:
    pct = 100 * gold_coverage[slot] / len(gold_ids)
    print(f'  {slot:16s} {gold_coverage[slot]:5d} / {len(gold_ids)}  ({pct:.1f}%)')

# MCL is read off COR_T1; Lateral/Medial Meniscus off SAG_FLUID_FS or SAG_FLUID_NOFS.
# If MCL-positive gold studies are missing COR_T1 at a meaningfully higher rate than
# MCL-negative ones, that's a real, fixable data-coverage bottleneck rather than a
# purely-harder-signal one.
mcl_positive_ids = set(train_labels.loc[gold_mask & (train_labels['MCL'] == 1), 'StudyInstanceUID'])
mcl_negative_ids = gold_ids - mcl_positive_ids
print(f'\nMCL-positive gold studies: {len(mcl_positive_ids)}, MCL-negative: {len(mcl_negative_ids)}')

for name, ids in [('MCL-positive', mcl_positive_ids), ('MCL-negative', mcl_negative_ids)]:
    subset = series_with_slots[series_with_slots['StudyInstanceUID'].isin(ids)]
    has_cor_t1 = 0
    for study_id, group in subset.groupby('StudyInstanceUID'):
        if slot_presence_mask(group['slot'].tolist())['COR_T1']:
            has_cor_t1 += 1
    pct = 100 * has_cor_t1 / len(ids) if ids else float('nan')
    print(f'  {name}: COR_T1 present in {has_cor_t1}/{len(ids)} ({pct:.1f}%)')